# **Geographical population structure and Pairwise Average Fst between coastal and inland cohorts**

In [1]:
#!pip install -q malariagen_data
!pip install malariagen_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 21.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 54.2 MB/s eta 0

In [2]:
!pip install scikit-learn

In [3]:
import os
import re
import plotly
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import numpy as np

In [ ]:
#from google.colab import drive
#drive.mount("drive")

In [ ]:
#results_dir = 'drive/MyDrive/Colab Data/gm-analysis/pca-results'
#os.makedirs(results_dir, exist_ok=True)

In [4]:
import malariagen_data

In [5]:
#ag3 = malariagen_data.Ag3(results_cache=results_dir)
#ag3
ag3 = malariagen_data.Ag3(pre=True)
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
Results cache                         : None
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.10.0
Client location                       : District of Columbia, United States (Google Cloud us-other)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.10.0/Ag3.html

In [6]:
df_samples=ag3.sample_metadata(sample_query="country=='Gambia, The' and taxon == 'arabiensis' and unrestricted_use == True")

In [7]:
arab_2019_sample_query = (
    "country=='Gambia, The' and "
    "taxon=='arabiensis' and "
    "sample_set=='1272-VO-GM-OPONDO-VMF00160' and "
    "sample_id!='VBS47740-6366STDY9854349'"
)

In [ ]:
# access metadata
df_arab_2019_sample = ag3.sample_metadata(sample_query=arab_2019_sample_query)
df_arab_2019_sample

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VBS47723-6366STDY9854332,C0454,Kevin Opondo,"Gambia, The",Dongoroba,2019,9,13.380,-15.294,F,...,Mansakonko,GM-L,Jarra East,arabiensis,GM-L_arab_2019,GM-L_arab_2019_09,GM-L_arab_2019_Q3,GM-L_Jarra-East_arab_2019,GM-L_Jarra-East_arab_2019_09,GM-L_Jarra-East_arab_2019_Q3
1,VBS47725-6366STDY9854334,C2409,Kevin Opondo,"Gambia, The",Kerewan,2019,9,13.515,-14.896,F,...,Janjanbureh,GM-M,Lower Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_09,GM-M_arab_2019_Q3,GM-M_Lower-Fuladu-West_arab_2019,GM-M_Lower-Fuladu-West_arab_2019_09,GM-M_Lower-Fuladu-West_arab_2019_Q3
2,VBS47726-6366STDY9854335,C0587,Kevin Opondo,"Gambia, The",Bansang,2019,9,13.434,-14.655,F,...,Janjanbureh,GM-M,Upper Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_09,GM-M_arab_2019_Q3,GM-M_Upper-Fuladu-West_arab_2019,GM-M_Upper-Fuladu-West_arab_2019_09,GM-M_Upper-Fuladu-West_arab_2019_Q3
3,VBS47729-6366STDY9854338,C2710,Kevin Opondo,"Gambia, The",Boiram,2019,10,13.503,-14.926,F,...,Janjanbureh,GM-M,Lower Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_10,GM-M_arab_2019_Q4,GM-M_Lower-Fuladu-West_arab_2019,GM-M_Lower-Fuladu-West_arab_2019_10,GM-M_Lower-Fuladu-West_arab_2019_Q4
4,VBS47731-6366STDY9854340,C2524,Kevin Opondo,"Gambia, The",Bansang,2019,9,13.434,-14.655,F,...,Janjanbureh,GM-M,Upper Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_09,GM-M_arab_2019_Q3,GM-M_Upper-Fuladu-West_arab_2019,GM-M_Upper-Fuladu-West_arab_2019_09,GM-M_Upper-Fuladu-West_arab_2019_Q3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309,VBS48167-6366STDY9854768,W1491,Kevin Opondo,"Gambia, The",Essau,2019,10,13.481,-16.515,F,...,Kerewan,GM-N,Lower Niumi,arabiensis,GM-N_arab_2019,GM-N_arab_2019_10,GM-N_arab_2019_Q4,GM-N_Lower-Niumi_arab_2019,GM-N_Lower-Niumi_arab_2019_10,GM-N_Lower-Niumi_arab_2019_Q4
310,VBS48171-6366STDY9854772,W2321,Kevin Opondo,"Gambia, The",Brikama-A,2019,9,13.290,-16.657,F,...,Brikama,GM-W,Kombo Central,arabiensis,GM-W_arab_2019,GM-W_arab_2019_09,GM-W_arab_2019_Q3,GM-W_Kombo-Central_arab_2019,GM-W_Kombo-Central_arab_2019_09,GM-W_Kombo-Central_arab_2019_Q3
311,VBS48173-6366STDY9854774,W0382,Kevin Opondo,"Gambia, The",Tallinding,2019,9,13.437,-16.670,F,...,Kanifing,GM-W,Jeshwang,arabiensis,GM-W_arab_2019,GM-W_arab_2019_09,GM-W_arab_2019_Q3,GM-W_Jeshwang_arab_2019,GM-W_Jeshwang_arab_2019_09,GM-W_Jeshwang_arab_2019_Q3
312,VBS48174-6366STDY9854775,W1168,Kevin Opondo,"Gambia, The",Tallinding,2019,9,13.437,-16.670,F,...,Kanifing,GM-W,Jeshwang,arabiensis,GM-W_arab_2019,GM-W_arab_2019_09,GM-W_arab_2019_Q3,GM-W_Jeshwang_arab_2019,GM-W_Jeshwang_arab_2019_09,GM-W_Jeshwang_arab_2019_Q3


In [ ]:
df_arab_2019_sample['cohort_admin1_year'].unique()

array(['GM-L_arab_2019', 'GM-M_arab_2019', 'GM-U_arab_2019',
       'GM-W_arab_2019', 'GM-N_arab_2019'], dtype=object)

In [ ]:
df_arab_2019_sample.groupby(['sample_set','admin1_name','cohort_admin1_year','cohort_admin2_year']).size()

sample_set                  admin1_name  cohort_admin1_year  cohort_admin2_year              
1272-VO-GM-OPONDO-VMF00160  Basse        GM-U_arab_2019      GM-U_Basse_arab_2019                28
                                                             GM-U_Jimara_arab_2019               37
                                                             GM-U_Sandu_arab_2019                 8
                                                             GM-U_Tumana_arab_2019               21
                                                             GM-U_Wuli-East_arab_2019             1
                            Brikama      GM-W_arab_2019      GM-W_Foni-Brefet_arab_2019           1
                                                             GM-W_Kombo-Central_arab_2019        28
                            Janjanbureh  GM-M_arab_2019      GM-M_Lower-Fuladu-West_arab_2019    51
                                                             GM-M_Niamina-East_arab_2019         10
                                                             GM-M_Niamina-West_arab_2019          8
                                                             GM-M_Upper-Fuladu-West_arab_2019    12
                            Kanifing     GM-W_arab_2019      GM-W_Bakau_arab_2019                 6
                                                             GM-W_Jeshwang_arab_2019             77
                            Kerewan      GM-N_arab_2019      GM-N_Illiasa_arab_2019               3
                                                             GM-N_Lower-Niumi_arab_2019          13
                            Kuntaur      GM-M_arab_2019      GM-M_Niani_arab_2019                 4
                            Mansakonko   GM-L_arab_2019      GM-L_Jarra-East_arab_2019            6
dtype: int64

### **Geographical population structure**

In [ ]:
# parameters to use in the PCA computations
region = "3L"
n_snps = 100_000
arab_sample_query = (
    "taxon == 'arabiensis' and "
    "country == 'Gambia, The' and "
    "sample_set == '1272-VO-GM-OPONDO-VMF00160' and "
    "sample_id != 'VBS47740-6366STDY9854349'"
)
site_mask = "arab"

In [ ]:
#pca = PCA(n_components=20)

In [ ]:
pca_df_arab, evr_arab = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=arab_sample_query,
    site_mask=site_mask,
    min_minor_ac=2,
    max_missing_an=0
    )

Compute SNP allele counts:   0%|          | 0/6393 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/7209 [00:00<?, ?it/s]

**Color by collection location**

In [ ]:
ag3.plot_pca_variance(evr_arab)

In [ ]:
# remove outliers !!! "VBS47740-6366STDY9854349" is from #2001
arab_outliers = [
    "VBS47740-6366STDY9854349", ##2001 gambia arab
    "VBS48175-6366STDY9854776",
    "VBS47997-6366STDY9854595",
    "VBS48073-6366STDY9854673",
    "VBS48085-6366STDY9854685",
    "VBS48029-6366STDY9854629",
    "VBS47764-6366STDY9854373",
    "VBS47995-6366STDY9854593",
    "VBS48055-6366STDY9854655",
    "VBS47767-6366STDY9854376",
    "VBS48013-6366STDY9854611",
    # "VBS48028-6366STDY9854628",
    "VBS48131-6366STDY9854733",
    "VBS47999-6366STDY9854597",
    "VBS47900-6366STDY9854498",
    "VBS47896-6366STDY9854495",
    # "VBS48133-6366STDY9854735",
    "VBS47775-6366STDY9854384",
    "VBS48126-6366STDY9854728",
    # "VBS48058-6366STDY9854658",
]

In [ ]:
region = "3L"
n_snps = 100_000

pca_df_arab_wo_outliers, evr_arab_wo_outliers = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=f"taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and sample_id not in {arab_outliers}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

Compute SNP allele counts:   0%|          | 0/6393 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/7209 [00:00<?, ?it/s]

### **Supplementary Figure 1. Proportion of genetic variance explained by the first 20 principal components**

In [ ]:
ag3.plot_pca_variance(evr_arab_wo_outliers)

In [ ]:
print(evr_arab_wo_outliers[0] * 100)

0.7486076


### **Supplementary Figure 2A. Population structure**

**Coloring by administrative division**

In [ ]:
# Define color mapping for each admin2 unit
color_mapping_admin2 = {
    'Jarra East': '#8e8c84',
    'Lower Fuladu West': '#7f7f7f',
    'Upper Fuladu West': '#6b705c',
    'Niani': '#4a4e4d',
    'Niamina East': '#3b5c40',
    'Jimara': '#8e7d32',
    'Basse': '#a95b29',
    'Wuli East': '#b64d3b',
    'Tumana': '#8a4b39',
    'Niamina West': '#6f4e37',
    'Sandu': '#3e3a3a',
    'Jeshwang': '#a4c9e2',
    'Kombo Central': '#78aaff',
    'Lower Niumi': '#4d88ff',
    'Bakau': '#0077b6',
    'Illiasa': '#0096c7',
    'Foni Brefet': '#00b4d8'
}

In [ ]:
# Apply color mapping to the PCA DataFrame
pca_df_arab_wo_outliers['color_admin2'] = pca_df_arab_wo_outliers['admin2_name'].map(color_mapping_admin2)

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    color='admin2_name',
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    y="PC3",
    x="PC1",
    color='admin2_name',
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

**Coloring by latitude/longitude coordinates**

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    x="longitude",
    y="PC1",
    color='admin2_name',
    labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

**Flipping the Axis to match the geographies of The Gambia**

In [ ]:
print(pca_df_arab_wo_outliers)

                    sample_id        PC1        PC2        PC3        PC4  \
0    VBS47723-6366STDY9854332 -15.574739 -10.290522   1.005367   8.018106   
1    VBS47725-6366STDY9854334 -29.463022  16.666611 -13.845028   4.498345   
2    VBS47726-6366STDY9854335 -32.548393  -7.278450  16.089140  10.171919   
3    VBS47729-6366STDY9854338 -37.592819  34.577698  18.094067 -15.912996   
4    VBS47731-6366STDY9854340 -37.217518   2.248578  21.159843 -18.428692   
..                        ...        ...        ...        ...        ...   
293  VBS48165-6366STDY9854766  40.991936  -9.585222  -2.014098  -2.967798   
294  VBS48167-6366STDY9854768  35.904594 -25.340012   5.971238 -29.179829   
295  VBS48171-6366STDY9854772  33.592567 -53.791477  26.402733 -27.791197   
296  VBS48173-6366STDY9854774  44.005806   4.764619  -4.854803   0.253644   
297  VBS48174-6366STDY9854775  62.089535 -34.370159 -11.440298 -25.376963   

           PC5        PC6         PC7        PC8         PC9  ...       tax

In [ ]:
# flip PC1 to switch the clusters
pca_df_arab_wo_outliers['PC1'] = -pca_df_arab_wo_outliers['PC1']

In [ ]:
pca_df_arab_wo_outliers[["PC1"]]

,PC1
0,15.574739
1,29.463022
2,32.548393
3,37.592819
4,37.217518
...,...
293,-40.991936
294,-35.904594
295,-33.592567
296,-44.005806


In [ ]:
pca_fig=ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    y="PC2",
    x="PC1",
    color='admin2_name',
    labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis",
    show=False
)

In [ ]:
pca_fig

### **Figure 1b. Geographical population structure**

**Cluster annotation - kmeans**

In [ ]:
df_samples_gambia.median_cov.min()

np.int64(10)

In [ ]:
df_samples_gambia.median_cov.max()

np.int64(53)

In [ ]:
df_samples_gambia.median_cov.mean()

np.float64(29.847619047619048)

In [ ]:
kmeans = KMeans(n_clusters=2, init = 'k-means++', max_iter=200)

In [ ]:
df_columns = ['PC1', 'PC2']

In [ ]:
km = kmeans.fit(pca_df_arab_wo_outliers[df_columns])

In [ ]:
centroids = km.cluster_centers_

In [ ]:
# Find assignment by cluster
data_pred = kmeans.fit_predict(pca_df_arab_wo_outliers[df_columns])
pca_df_arab_wo_outliers['cluster'] = data_pred

In [ ]:
# Split data by group for generated bounds for ellipse
data0 = pca_df_arab_wo_outliers[pca_df_arab_wo_outliers['cluster']==0]
data1 = pca_df_arab_wo_outliers[pca_df_arab_wo_outliers['cluster']==1]

In [ ]:
# Calculate covariance matrix
cov_matrix = np.cov(data0['PC1'], data0['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
# Determine the angle from eigenvectors
angle = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
# Get center and scale of ellipse
center = np.mean(np.array([data0['PC1'], data0['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor = np.sqrt(5.991)
radii = scaling_factor * np.sqrt(eigenvalues)

t = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto = center[0] + radii[0] * np.cos(t) * np.cos(angle) - radii[1] * np.sin(t) * np.sin(angle)
ellipse_y_auto = center[1] + radii[0] * np.cos(t) * np.sin(angle) + radii[1] * np.sin(t) * np.cos(angle)

In [ ]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto,
    y=ellipse_y_auto,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#a4c9e2', width=2)
    #fill='toself' # Fills the area enclosed by the ellipse
    ))


pca_fig.show()

In [ ]:
# Calculate covariance matrix
cov_matrix1 = np.cov(data1['PC1'], data1['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues1, eigenvectors1 = np.linalg.eig(cov_matrix1)
# Determine the angle from eigenvectors
angle1 = np.arctan2(eigenvectors1[1, 0], eigenvectors1[0, 0])
# Get center and scale of ellipse
center1 = np.mean(np.array([data1['PC1'], data1['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor1 = np.sqrt(5.991)
radii1 = scaling_factor1 * np.sqrt(eigenvalues1)

t1 = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto1 = center1[0] + radii1[0] * np.cos(t1) * np.cos(angle1) - radii1[1] * np.sin(t1) * np.sin(angle1)
ellipse_y_auto1 = center1[1] + radii1[0] * np.cos(t1) * np.sin(angle1) + radii1[1] * np.sin(t1) * np.cos(angle1)

In [ ]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto1,
    y=ellipse_y_auto1,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#b64d3b', width=2),
    #fill='toself' # Fills the area enclosed by the ellipse
    ))


pca_fig.show()

In [ ]:
pca_fig.add_trace(go.Scatter(
    x=centroids[:,0],
    y=centroids[:,1],
    mode="text",
    name="Centroids",
    text=["IC", "CC"],
    textposition="top center"
    ))
pca_fig.update_traces(textfont_size=20)

In [ ]:
admin_labels=df_samples.admin2_name.unique()

In [ ]:
for trace in pca_fig['data']:
    if (not trace['name'] in admin_labels):
        trace['showlegend'] = False

In [ ]:
pca_fig

### **Supplementary Figure 2B. Neighbour joining tree**

In [ ]:
ag3.plot_njt(
    region="3L",
    n_snps=50_000,
    site_mask="arab",
    sample_query=f"taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and sample_id not in {arab_outliers}",
    color="admin2_name",
    #labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2
)

Compute SNP allele counts:   0%|          | 0/6393 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/7209 [00:00<?, ?it/s]

Construct neighbour-joining tree:   0%|          | 0/296 [00:00<?, ?it/s]

### **Supplementary Figure 3. Population structure - The Gambia and reference populations WA**

In [8]:
WA_sample_sets = ['AG1000G-BF-B','AG1000G-CM-C','1272-VO-GM-OPONDO-VMF00160','AG1000G-ML-B','1314-VO-BF-KIENTEGA-KIMA-BF-2104']

In [9]:
df_WA__samples=ag3.sample_metadata(sample_query=f"sample_set in {WA_sample_sets} and taxon == 'arabiensis' and sample_id!='VBS47740-6366STDY9854349'")

In [10]:
df_WA__samples.groupby('country').size()

,0
country,
Burkina Faso,55
Cameroon,2
"Gambia, The",314
Mali,2


In [11]:
region = "3L:15,000,000-41,000,000"
n_snps = 100_000

In [14]:
outlier_list=['VBS47997-6366STDY9854595',
              'VBS47740-6366STDY9854349', #2001 gambia arab
 'VBS48073-6366STDY9854673',
 'VBS48085-6366STDY9854685',
 'VBS48175-6366STDY9854776',
#round2
 'VBS47995-6366STDY9854593', 'VBS48029-6366STDY9854629'#
#round3
'VBS47764-6366STDY9854373',
 'VBS47767-6366STDY9854376',
 'VBS47999-6366STDY9854597',
 'VBS48013-6366STDY9854611',
 'VBS48028-6366STDY9854628',
 'VBS48055-6366STDY9854655',
 'VBS48131-6366STDY9854733'
 #round4
'VBS47764-6366STDY9854373', 'VBS48029-6366STDY9854629',
#round5
'VBS47764-6366STDY9854373','VBS48131-6366STDY9854733']

In [16]:
color_mapping_admin2_refs = {
    'Jarra East': '#8e8c84',
    'Lower Fuladu West': '#7f7f7f',
    'Upper Fuladu West': '#6b705c',
    'Niani': '#4a4e4d',
    'Niamina East': '#3b5c40',
    'Jimara': '#8e7d32',
    'Basse': '#a95b29',
    'Wuli East': '#b64d3b',
    'Tumana': '#8a4b39',
    'Niamina West': '#6f4e37',
    'Sandu': '#3e3a3a',
    'Jeshwang': '#a4c9e2',
    'Kombo Central': '#78aaff',
    'Lower Niumi': '#4d88ff',
    'Bakau': '#0077b6',
    'Illiasa': '#0096c7',
    'Foni Brefet': '#00b4d8',
    'Houet': '#ff0072',
    'Benoue': '#e673af',
    'Bla': '#c798ef',
    'Baroueli': '#c798ef'
}

In [17]:
pca_df, evr = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=f"sample_set in {WA_sample_sets} and taxon == 'arabiensis'",
    exclude_samples=outlier_list,
);

Compute SNP allele counts:   0%|          | 0/8085 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/8773 [00:00<?, ?it/s]

In [18]:
#round5 outlier removal
ag3.plot_pca_coords(
    pca_df,
    color="admin2_name",
    labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2_refs,
)

In [19]:
pca_fig=ag3.plot_pca_coords(
    pca_df,
    color="admin2_name",
    labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2_refs,
    show=False,
)

In [20]:
from sklearn.cluster import KMeans
import numpy as np
import plotly.graph_objects as go

In [21]:
kmeans = KMeans(n_clusters=3, init = 'k-means++', max_iter=200)

In [28]:
df_columns = ['PC1', 'PC2']

In [29]:
# Find assignment by cluster
data_pred = kmeans.fit_predict(pca_df[df_columns])
pca_df['cluster'] = data_pred

In [30]:
# Split data by group for generated bounds for ellipse
data0 = pca_df[pca_df['cluster']==0]
data1 = pca_df[pca_df['cluster']==1]
data2 = pca_df[pca_df['cluster']==2]

In [31]:
# Calculate covariance matrix
cov_matrix = np.cov(data0['PC1'], data0['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
# Determine the angle from eigenvectors
angle = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
# Get center and scale of ellipse
center = np.mean(np.array([data0['PC1'], data0['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor = np.sqrt(5.991)
radii = scaling_factor * np.sqrt(eigenvalues)

t = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto = center[0] + radii[0] * np.cos(t) * np.cos(angle) - radii[1] * np.sin(t) * np.sin(angle)
ellipse_y_auto = center[1] + radii[0] * np.cos(t) * np.sin(angle) + radii[1] * np.sin(t) * np.cos(angle)

In [32]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto,
    y=ellipse_y_auto,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#b64d3b', width=2)
    #fill='toself' # Fills the area enclosed by the ellipse
    ))

In [33]:
# Calculate covariance matrix
cov_matrix1 = np.cov(data1['PC1'], data1['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues1, eigenvectors1 = np.linalg.eig(cov_matrix1)
# Determine the angle from eigenvectors
angle1 = np.arctan2(eigenvectors1[1, 0], eigenvectors1[0, 0])
# Get center and scale of ellipse
center1 = np.mean(np.array([data1['PC1'], data1['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor1 = np.sqrt(5.991)
radii1 = scaling_factor1 * np.sqrt(eigenvalues1)

t1 = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto1 = center1[0] + radii1[0] * np.cos(t1) * np.cos(angle1) - radii1[1] * np.sin(t1) * np.sin(angle1)
ellipse_y_auto1 = center1[1] + radii1[0] * np.cos(t1) * np.sin(angle1) + radii1[1] * np.sin(t1) * np.cos(angle1)

In [34]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto1,
    y=ellipse_y_auto1,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#ff0072', width=2),
    #fill='toself' # Fills the area enclosed by the ellipse
    ))

In [35]:
# Calculate covariance matrix
cov_matrix2 = np.cov(data2['PC1'], data2['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues2, eigenvectors2 = np.linalg.eig(cov_matrix2)
# Determine the angle from eigenvectors
angle2 = np.arctan2(eigenvectors2[1, 0], eigenvectors2[0, 0])
# Get center and scale of ellipse
center2 = np.mean(np.array([data2['PC1'], data2['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor2 = np.sqrt(5.991)
radii2 = scaling_factor2 * np.sqrt(eigenvalues2)

t2 = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto2 = center2[0] + radii2[0] * np.cos(t2) * np.cos(angle2) - radii2[1] * np.sin(t2) * np.sin(angle2)
ellipse_y_auto2 = center2[1] + radii2[0] * np.cos(t2) * np.sin(angle2) + radii2[1] * np.sin(t2) * np.cos(angle2)

In [36]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto2,
    y=ellipse_y_auto2,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#a4c9e2', width=2),
    #fill='toself' # Fills the area enclosed by the ellipse
    ))

In [37]:
admin_labels=df_WA__samples.admin2_name.unique()

In [38]:
for trace in pca_fig['data']:
    if (not trace['name'] in admin_labels):
        trace['showlegend'] = False

In [39]:
pca_fig

In [40]:
df_WA__samples.groupby('country').size()

,0
country,
Burkina Faso,55
Cameroon,2
"Gambia, The",314
Mali,2


### **Pairwise Average Fst between coastal and inland cohorts**

Compute pairwise average Hudson’s Fst between a set of specified cohorts

### **Figure 1c. Fst analysis**

In [ ]:
#define some useful variables

#lists of chromosome/arms for analyses
chrom_arms = ['2R', '2L', '3R','3L','X']
chroms = ['2RL', '3RL','X']

#dict of  analysis groupings
admin1_dict = {
    'Basse' : "admin2_name == 'Basse' | admin2_name == 'Jimara' | admin2_name == 'Sandu' | admin2_name == 'Tumana' | admin2_name == 'Wuli East'",
    'Brikama' : "admin2_name == 'Foni Brefet' | admin2_name == 'Kombo Central'",
    'Janjanbureh' : "admin2_name == 'Lower Fuladu West' | admin2_name == 'Niamina East' | admin2_name == 'Niamina West' | admin2_name == 'Upper Fuladu West'",
    'Kanifing' : "admin2_name == 'Bakau' | admin2_name == 'Jeshwang'",
    'Kerewan' : "admin2_name ==  'Illiasa' | admin2_name == 'Lower Niumi'",
    'Kuntaur' : "admin2_name == 'Niani'",
    'Mansakonko' : "admin2_name == 'Jarra East'"
}

admin2_dict = {
    'Basse' : "location == 'Kaba Kamma' | location == 'Sabi' | location == 'Santo Su'",
    'Jimara' : "location == 'Damba kunda Kuta' | location == 'Gambisara' | location == 'Kossemar' | location == 'Sotuma sere'",
    'Sandu' : "location == 'Sare Demba'",
    'Tumana' : "location == 'Nyakoi Madina' | location == 'Sare Alpha' | location == 'Tambansang' | location == 'Tuba Simoto'",
    'Wuli East' : "location == 'Baja Kunda'",
    'Foni Brefet' : "location == 'Bullock'",
    'Kombo Central' : "location == 'Brikama'",
    'Lower Fuladu West' : "location == 'Boiram' | location == 'Jakhali' | location == 'Kerewan' | location == 'Sankulay kunda' | location == 'Sapu' | location == 'Saruja' | location == 'Wali Kunda'",
    'Niamina East' : "location== 'Jareng' | location == 'Sare Silli'",
    'Niamina West' : "location == 'Kumbija'",
    'Upper Fuladu West' : "location == 'Bansang'",
    'Bakau' : "location == 'Bakau'",
    'Jeshwang' : "location == 'Ebotown', location == 'Tallinding'",
    'Illiasa' : "location ==  'Farafenni'",
    'Lower Niumi' : "location ==  'Essau'",
    'Niani' : "location == 'Kuntaur' | location == 'Nyanga Batang'",
    'Jarra East' : "location == 'Dongoroba'"
}

#dict of cohorts/locations for labelling
cohort_dict = {
    'GM-U_arab_2019' : 'Basse',
    'GM-W_arab_2019' : 'Brikama',
    'GM-M_arab_2019' : 'Janjanbureh',
    'GM-W_arab_2019' : 'Kanifing',
    'GM-N_arab_2019' : 'Kerewan',
    'GM-M_arab_2019' : 'Kuntaur',
    #'GM-L_arab_2001' : 'Mansakonko',
    'GM-L_arab_2019' : 'Mansakonko'
}

cohort_dict2 = {
    'GM-U_arab_2019' : 'Basse',
    'GM-U_arab_2019' : 'Jimara',
    'GM-U_arab_2019' : 'Sandu',
    'GM-U_arab_2019' : 'Tumana',
    'GM-U_arab_2019' : 'Wuli East',
    'GM-W_arab_2019' : 'Foni Brefet',
    'GM-W_arab_2019' : 'Kombo Central',
    'GM-M_arab_2019' : 'Lower Fuladu West',
    'GM-M_arab_2019' : 'Niamina East',
    'GM-M_arab_2019' : 'Niamina West',
    'GM-M_arab_2019' : 'Upper Fuladu West',
    'GM-W_arab_2019' : 'Bakau',
    'GM-W_arab_2019' : 'Jeshwang',
    'GM-N_arab_2019' : 'Illiasa',
    'GM-N_arab_2019' : 'Lower Niumi',
    'GM-M_arab_2019' : 'Niani',
    'GM-L_arab_2019' : 'Jarra East'
}

In [ ]:
fst_arab_df2 = ag3.pairwise_average_fst(
    region="3L",
    cohorts="admin2_name",
    sample_query=(
        "country == 'Gambia, The' and "
        "sample_set == '1272-VO-GM-OPONDO-VMF00160' and "
        "taxon == 'arabiensis' and "
        "sample_id != 'VBS47740-6366STDY9854349'" #2001 sample
    ),
    min_cohort_size=10,
    site_mask="arab",
)

fst_arab_df2

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/fst.py:496: UserWarning:

Cohort (Bakau) has insufficient samples (6) for requested cohort size (10), dropping.

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/fst.py:496: UserWarning:

Cohort (Foni Brefet) has insufficient samples (1) for requested cohort size (10), dropping.

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/fst.py:496: UserWarning:

Cohort (Illiasa) has insufficient samples (3) for requested cohort size (10), dropping.

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/fst.py:496: UserWarning:

Cohort (Jarra East) has insufficient samples (6) for requested cohort size (10), dropping.

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/fst.py:496: UserWarning:

Cohort (Niamina West) has insufficient samples (8) for requested cohort size (10), dropping.

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/fst.py:496: UserWarning:

Cohort (Niani) ha

Compute SNP allele counts:   0%|          | 0/1905 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/malariagen_data/anoph/snp_data.py:1190: UserWarning:

Cohort downsampled from 77 to 50 samples. Set max_cohort_size=None to disable downsampling.



Compute SNP allele counts:   0%|          | 0/2993 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1905 [00:00<?, ?it/s]

...

Compute SNP allele counts:   0%|          | 0/2313 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1905 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/2313 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1905 [00:00<?, ?it/s]

,cohort1,cohort2,fst,se
0,Basse,Jeshwang,0.007823,0.000360
1,Basse,Jimara,0.000286,0.000122
2,Basse,Kombo Central,0.003858,0.000214
3,Basse,Lower Fuladu West,0.000000,0.000090
4,Basse,Lower Niumi,0.006732,0.000408
5,Basse,Niamina East,0.000000,0.000198
6,Basse,Tumana,0.000000,0.000152
7,Basse,Upper Fuladu West,0.000000,0.000220
8,Jeshwang,Jimara,0.007948,0.000330
9,Jeshwang,Kombo Central,0.002581,0.000213


In [ ]:
ag3.plot_pairwise_average_fst(fst_arab_df2)

In [ ]:
#Fst between cohorts of admin1
import itertools as it
chroms = ['2RL','3RL','X']
admin1_locs = ['Basse', 'Brikama', 'Janjanbureh', 'Kanifing', 'Kerewan', 'Kuntaur', 'Mansakonko']

fst_df_list = []
for region_a,region_b in it.combinations(admin1_locs,2):
    for chrom in chroms:
        fstar = ag3.fst_gwss(contig = f'{chrom}', min_cohort_size=10, window_size=5000, cohort1_query=admin1_dict[f'{region_a}'], cohort2_query=admin1_dict[f'{region_b}'])
        fst_df = pd.DataFrame({'pos':fstar[0], 'fst':fstar[1]})
        fst_df['chrom'] = chrom
        fst_df['admin1_a'] = region_a
        fst_df['admin1_b'] = region_b
        fst_df_list.append(fst_df)

    fst_bigdf = pd.concat(fst_df_list)

#fst_bigdf.to_csv('./Fst-analysis-results/fstdf_admin1.csv')

In [ ]:
#Fst between cohorts of admin2
import itertools as it
chroms = ['2RL','3RL','X']
admin2_locs = ['Basse', 'Jimara', 'Sandu', 'Tumana', 'Wuli East', 'Foni Brefet', 'Kombo Central', 'Lower Fuladu West', 'Niamina East',
               'Niamina West', 'Upper Fuladu West', 'Bakau', 'Jeshwang', 'Illiasa', 'Lower Niumi', 'Niani', 'Jarra East']
